In [1]:
# Install and set up PySpark in Google Colab
#Spark Aggregate : Group rows in DataFrame, Get Count distinct on DataFrame, Add row number to
#DataFrame, Select the first row of each group
!pip install pyspark

# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create SparkSession
spark = SparkSession.builder \
    .appName("Spark Aggregations Example") \
    .getOrCreate()

# Sample data
data = [
    (1, "Alice", "HR", 3000),
    (2, "Bob", "HR", 3500),
    (3, "Charlie", "IT", 4000),
    (4, "David", "IT", 4500),
    (5, "Eve", "IT", 5000),
    (6, "Frank", "Finance", 2800),
    (7, "Grace", "Finance", 3200),
    (8, "Hannah", "HR", 3700),
]

# Create DataFrame
columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|     HR|  3000|
|  2|    Bob|     HR|  3500|
|  3|Charlie|     IT|  4000|
|  4|  David|     IT|  4500|
|  5|    Eve|     IT|  5000|
|  6|  Frank|Finance|  2800|
|  7|  Grace|Finance|  3200|
|  8| Hannah|     HR|  3700|
+---+-------+-------+------+



In [2]:
# 1. Aggregate: Group rows in DataFrame (e.g., total salary per department)
print("\nTotal Salary per Department:")
df.groupBy("dept").agg(F.sum("salary").alias("total_salary")).show()



Total Salary per Department:
+-------+------------+
|   dept|total_salary|
+-------+------------+
|     HR|       10200|
|     IT|       13500|
|Finance|        6000|
+-------+------------+



In [3]:
# 2. Count distinct on DataFrame (distinct departments)
print("\nCount of Distinct Departments:")
distinct_dept_count = df.select(F.countDistinct("dept").alias("distinct_dept_count"))
distinct_dept_count.show()


Count of Distinct Departments:
+-------------------+
|distinct_dept_count|
+-------------------+
|                  3|
+-------------------+



In [4]:
# 3. Add row number to DataFrame
print("\nDataFrame with Row Number:")
window_spec_all = Window.orderBy("id")
df_with_rownum = df.withColumn("row_number", F.row_number().over(window_spec_all))
df_with_rownum.show()


DataFrame with Row Number:
+---+-------+-------+------+----------+
| id|   name|   dept|salary|row_number|
+---+-------+-------+------+----------+
|  1|  Alice|     HR|  3000|         1|
|  2|    Bob|     HR|  3500|         2|
|  3|Charlie|     IT|  4000|         3|
|  4|  David|     IT|  4500|         4|
|  5|    Eve|     IT|  5000|         5|
|  6|  Frank|Finance|  2800|         6|
|  7|  Grace|Finance|  3200|         7|
|  8| Hannah|     HR|  3700|         8|
+---+-------+-------+------+----------+



In [5]:
# 4. Select the first row of each group (e.g., first employee in each department by salary descending)
print("\nFirst Row of Each Department by Highest Salary:")
window_spec_dept = Window.partitionBy("dept").orderBy(F.desc("salary"))
first_row_per_dept = df.withColumn("row_number", F.row_number().over(window_spec_dept)) \
                       .filter(F.col("row_number") == 1) \
                       .drop("row_number")
first_row_per_dept.show()


First Row of Each Department by Highest Salary:
+---+------+-------+------+
| id|  name|   dept|salary|
+---+------+-------+------+
|  7| Grace|Finance|  3200|
|  8|Hannah|     HR|  3700|
|  5|   Eve|     IT|  5000|
+---+------+-------+------+



In [6]:
# Stop SparkSession
spark.stop()
